# P3 — Statistical Significance: 3-Seed Training
## Notebook 06: `06_statistical_significance.ipynb`

**Project:** Optimizing Inference Latency in Enterprise NLP via Task-Specific Knowledge Distillation  
**Group 15 | Section 2241044 | ITER, Siksha 'O' Anusandhan University**

---

### Purpose of This Notebook

A single training run is not statistically defensible in a research paper.  
Random weight initialisation and data shuffling introduce variance — two researchers  
running the same code can get different results.

This notebook runs **both models (KD pipeline and vanilla baseline) 3 times each**  
with different random seeds and reports `mean ± std` for all metrics.

| Model | Seeds | Runs |
|---|---|---|
| KD Pipeline (pseudo-labels) | 42, 7, 123 | 3 |
| Vanilla Baseline (gold labels) | 42, 7, 123 | 3 |

**Expected runtime: ~60 minutes total on T4 GPU.**  
All results auto-save to Drive after each run — safe to resume if interrupted.

---

### What This Produces

```
KD Pipeline    : Macro F1 = 0.XXX ± 0.XXX  (mean ± std across 3 seeds)
Vanilla Base   : Macro F1 = 0.XXX ± 0.XXX
```

This is the format required for all ML research paper result tables.

---

### Cell 1 — Restore Session

Reinstalls libraries, mounts Drive, loads all required data splits and pseudo-labels.  
Sets up label maps identical to all previous notebooks.

In [5]:
!pip install transformers scikit-learn pandas numpy torch -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os

# ── MASTER DRIVE PATH ─────────────────────────────────────────
# All data lives in ejdotp@gmail.com's Drive
# When logged in as ejdotp        → files are in MyDrive/KD_Project
# When logged in as any other acc → files are in Shared with me/KD_Project

MASTER_PATH = '/content/drive/MyDrive/KD_Project'
SHARED_PATH = '/content/drive/Shareddrives/KD_Project'
ALT_PATH    = '/content/drive/MyDrive/KD_Project'  # fallback

if os.path.exists(MASTER_PATH):
    BASE = MASTER_PATH
    print("Logged in as master account (ejdotp). Using MyDrive.")
elif os.path.exists(SHARED_PATH):
    BASE = SHARED_PATH
    print("Logged in as secondary account. Using Shared Drive.")
else:
    BASE = ALT_PATH
    print(f"WARNING: Could not auto-detect path. Using fallback: {BASE}")

print(f"Base path : {BASE}")

# ── LOAD ALL SPLITS ───────────────────────────────────────────
train_df  = pd.read_csv(f'{BASE}/train.csv')
val_df    = pd.read_csv(f'{BASE}/val.csv')
test_df   = pd.read_csv(f'{BASE}/test.csv')
pseudo_df = pd.read_csv(f'{BASE}/pseudo_labels_final.csv')

# Label maps
SENTIMENT_MAP = {"negative": 0, "neutral": 1, "positive": 2}
URGENCY_MAP   = {"non-urgent": 0, "urgent": 1}
INV_SENTIMENT = {v: k for k, v in SENTIMENT_MAP.items()}
INV_URGENCY   = {v: k for k, v in URGENCY_MAP.items()}

# Apply label maps
train_df["sentiment_id"]  = train_df["sentiment"].map(SENTIMENT_MAP)
val_df["sentiment_id"]    = val_df["sentiment"].map(SENTIMENT_MAP)
test_df["sentiment_id"]   = test_df["sentiment"].map(SENTIMENT_MAP)
pseudo_df["sentiment_id"] = pseudo_df["pseudo_sentiment"].map(SENTIMENT_MAP)
pseudo_df["urgency_id"]   = pseudo_df["pseudo_urgency"].map(URGENCY_MAP)

pseudo_df = pseudo_df.dropna(subset=["sentiment_id", "urgency_id"]).reset_index(drop=True)
pseudo_df["sentiment_id"] = pseudo_df["sentiment_id"].astype(int)
pseudo_df["urgency_id"]   = pseudo_df["urgency_id"].astype(int)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Train (gold)   : {len(train_df)} samples")
print(f"Train (pseudo) : {len(pseudo_df)} samples")
print(f"Val            : {len(val_df)} samples")
print(f"Test           : {len(test_df)} samples")
print(f"Device         : {device}")
print(f"GPU            : {torch.cuda.get_device_name(0)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Logged in as master account (ejdotp). Using MyDrive.
Base path : /content/drive/MyDrive/KD_Project
Train (gold)   : 3876 samples
Train (pseudo) : 3876 samples
Val            : 485 samples
Test           : 485 samples
Device         : cuda
GPU            : Tesla T4


### Cell 2 — Seed Control Utility

Defines a `set_seed()` function that fixes all sources of randomness:
- Python random
- NumPy random
- PyTorch CPU and GPU random states
- CuDNN deterministic mode

This ensures each seed produces a **reproducible but distinct** training run.  
Without this, results across runs would vary unpredictably.

In [6]:
import random
from transformers import DistilBertTokenizerFast

def set_seed(seed: int):
    """
    Fix all randomness sources for reproducible training.
    Must be called before model initialisation and DataLoader creation.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f"Seed set to {seed}.")

# Load tokenizer once — shared across all runs
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
print("Tokenizer loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded.


### Cell 3 — Dataset Classes

Defines two Dataset classes:
- `DualTaskDataset` — for the KD pipeline (sentiment + urgency labels)
- `SingleTaskDataset` — for the vanilla baseline (sentiment only)

Both use the same tokenizer and max_length=128, consistent with all previous notebooks.

In [7]:
from torch.utils.data import Dataset, DataLoader

class DualTaskDataset(Dataset):
    """For KD pipeline — sentiment + urgency pseudo-labels."""
    def __init__(self, texts, sentiment_labels, urgency_labels,
                 tokenizer, max_length=128):
        self.encodings = tokenizer(
            list(texts),
            truncation=True, padding=True,
            max_length=max_length, return_tensors='pt'
        )
        self.sentiment_labels = torch.tensor(sentiment_labels, dtype=torch.long)
        self.urgency_labels   = torch.tensor(urgency_labels,   dtype=torch.long)

    def __len__(self):
        return len(self.sentiment_labels)

    def __getitem__(self, idx):
        return {
            'input_ids'      : self.encodings['input_ids'][idx],
            'attention_mask' : self.encodings['attention_mask'][idx],
            'sentiment_label': self.sentiment_labels[idx],
            'urgency_label'  : self.urgency_labels[idx]
        }


class SingleTaskDataset(Dataset):
    """For vanilla baseline — gold sentiment labels only."""
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            list(texts),
            truncation=True, padding=True,
            max_length=max_length, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids'     : self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label'         : self.labels[idx]
        }

print("Dataset classes defined.")

Dataset classes defined.


### Cell 4 — Model Architectures

Defines both model classes exactly as used in notebooks 03 and 05:
- `DualHeadDistilBERT` — KD pipeline model (sentiment + urgency heads)
- `VanillaDistilBERT` — baseline model (sentiment only)

Both are re-initialised fresh for each seed run.

In [8]:
from transformers import DistilBertModel

class DualHeadDistilBERT(nn.Module):
    """KD pipeline model — dual classification heads."""
    def __init__(self, num_sentiment=3, num_urgency=2, dropout=0.3):
        super(DualHeadDistilBERT, self).__init__()
        self.distilbert     = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout        = nn.Dropout(dropout)
        self.sentiment_head = nn.Linear(768, num_sentiment)
        self.urgency_head   = nn.Linear(768, num_urgency)

    def forward(self, input_ids, attention_mask):
        outputs    = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        return self.sentiment_head(cls_output), self.urgency_head(cls_output)


class VanillaDistilBERT(nn.Module):
    """Vanilla baseline model — single classification head."""
    def __init__(self, num_classes=3, dropout=0.3):
        super(VanillaDistilBERT, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs    = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        return self.classifier(cls_output)

print("Model architectures defined.")

Model architectures defined.


### Cell 5 — Class Weight Calculator

Computes class weights for both tasks.  
Encapsulated as a function so it can be called cleanly inside each seed run.

**Sentiment weights** — based on gold label distribution in train split.  
**Urgency weights** — based on pseudo-label distribution (29:1 imbalance).

In [9]:
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_linear_schedule_with_warmup

def get_class_weights(labels, num_classes, device):
    """Compute balanced class weights and move to device."""
    weights = compute_class_weight(
        class_weight = 'balanced',
        classes      = np.array(range(num_classes)),
        y            = labels
    )
    return torch.tensor(weights, dtype=torch.float).to(device)

# Pre-compute — same values used across all seeds
sentiment_weights_vals = compute_class_weight(
    'balanced', classes=np.array([0,1,2]),
    y=train_df["sentiment_id"].values
)
urgency_weights_vals = compute_class_weight(
    'balanced', classes=np.array([0,1]),
    y=pseudo_df["urgency_id"].values
)

print("Sentiment class weights:")
for i, w in enumerate(sentiment_weights_vals):
    print(f"  {INV_SENTIMENT[i]:>10} : {w:.4f}")
print("\nUrgency class weights:")
for i, w in enumerate(urgency_weights_vals):
    print(f"  {INV_URGENCY[i]:>12} : {w:.4f}")

Sentiment class weights:
    negative : 2.6749
     neutral : 0.5610
    positive : 1.1853

Urgency class weights:
    non-urgent : 0.5172
        urgent : 15.0233


### Cell 6 — KD Pipeline Training Function

Encapsulates the complete KD training loop as a single callable function.  
Takes a `seed` parameter — everything else is fixed.

Identical hyperparameters to `03_distilbert_finetuning.ipynb`:
- lr = 2e-5, epochs = 5, batch_size = 16
- Combined loss: α=0.6 (sentiment) + β=0.4 (urgency)
- Gradient clipping at max_norm=1.0

Returns test Macro F1, per-class F1, and full classification report.

In [10]:
from sklearn.metrics import f1_score, classification_report
import time

EPOCHS = 5
ALPHA  = 0.6   # sentiment loss weight
BETA   = 0.4   # urgency loss weight

def train_kd_pipeline(seed: int):
    """
    Full KD pipeline training run for one seed.
    Returns dict of test metrics.
    """
    print(f"\n{'='*60}")
    print(f"KD PIPELINE — SEED {seed}")
    print(f"{'='*60}")

    set_seed(seed)

    # ── DataLoaders ───────────────────────────────────────────
    train_dataset = DualTaskDataset(
        pseudo_df["text"].values,
        pseudo_df["sentiment_id"].values,
        pseudo_df["urgency_id"].values,
        tokenizer
    )
    val_dataset = DualTaskDataset(
        val_df["text"].values,
        val_df["sentiment_id"].values,
        np.zeros(len(val_df), dtype=int),
        tokenizer
    )
    test_dataset = SingleTaskDataset(
        test_df["text"].values,
        test_df["sentiment_id"].values,
        tokenizer
    )
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

    # ── Model ─────────────────────────────────────────────────
    model = DualHeadDistilBERT().to(device)

    # ── Loss ──────────────────────────────────────────────────
    s_weights = torch.tensor(sentiment_weights_vals, dtype=torch.float).to(device)
    u_weights = torch.tensor(urgency_weights_vals,   dtype=torch.float).to(device)
    s_criterion = nn.CrossEntropyLoss(weight=s_weights)
    u_criterion = nn.CrossEntropyLoss(weight=u_weights)

    # ── Optimiser + Scheduler ─────────────────────────────────
    optimizer  = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler  = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = int(0.1 * total_steps),
        num_training_steps = total_steps
    )

    # ── Training loop ─────────────────────────────────────────
    best_val_f1      = 0.0
    best_model_state = None

    print(f"{'Epoch':<8}{'Train Loss':<14}{'Val F1':<12}{'Time'}")
    print("-" * 45)

    for epoch in range(EPOCHS):
        t0 = time.time()
        model.train()
        total_loss = 0

        for batch in train_loader:
            input_ids        = batch['input_ids'].to(device)
            attention_mask   = batch['attention_mask'].to(device)
            s_labels         = batch['sentiment_label'].to(device)
            u_labels         = batch['urgency_label'].to(device)

            optimizer.zero_grad()
            s_logits, u_logits = model(input_ids, attention_mask)
            loss = ALPHA * s_criterion(s_logits, s_labels) +                    BETA  * u_criterion(u_logits, u_labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        # Validation
        model.eval()
        all_preds, all_labels_v = [], []
        with torch.no_grad():
            for batch in val_loader:
                s_logits, _ = model(
                    batch['input_ids'].to(device),
                    batch['attention_mask'].to(device)
                )
                all_preds.extend(torch.argmax(s_logits, 1).cpu().numpy())
                all_labels_v.extend(batch['sentiment_label'].numpy())

        val_f1 = f1_score(all_labels_v, all_preds, average='macro')
        print(f"Epoch {epoch+1:<4} {total_loss/len(train_loader):<14.4f}{val_f1:<12.4f}{time.time()-t0:.1f}s")

        if val_f1 > best_val_f1:
            best_val_f1      = val_f1
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            print(f"         -> Best saved (Val F1: {val_f1:.4f})")

    # ── Test evaluation ───────────────────────────────────────
    model.load_state_dict(best_model_state)
    model.eval()
    all_preds, all_labels_t = [], []

    with torch.no_grad():
        for batch in test_loader:
            logits, _ = model(
                batch['input_ids'].to(device),
                batch['attention_mask'].to(device)
            )
            all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
            all_labels_t.extend(batch['label'].numpy())

    macro_f1     = f1_score(all_labels_t, all_preds, average='macro')
    per_class_f1 = f1_score(all_labels_t, all_preds, average=None)
    accuracy     = (np.array(all_preds) == np.array(all_labels_t)).mean()

    print(f"\nTest Macro F1 : {macro_f1:.4f}")
    print(f"Test Accuracy : {accuracy:.4f}")
    print(classification_report(
        all_labels_t, all_preds,
        target_names=["negative", "neutral", "positive"]
    ))

    return {
        "seed"        : seed,
        "model"       : "KD Pipeline",
        "macro_f1"    : round(macro_f1, 4),
        "accuracy"    : round(accuracy, 4),
        "negative_f1" : round(per_class_f1[0], 4),
        "neutral_f1"  : round(per_class_f1[1], 4),
        "positive_f1" : round(per_class_f1[2], 4),
        "best_val_f1" : round(best_val_f1, 4)
    }

print("KD training function defined.")

KD training function defined.


### Cell 7 — Vanilla Baseline Training Function

Same structure as Cell 6 but for the vanilla baseline:
- Uses gold labels from `train_df` instead of pseudo-labels
- Single-head model (`VanillaDistilBERT`)
- Single cross-entropy loss (no urgency task)

All other hyperparameters identical.

In [11]:
def train_vanilla_baseline(seed: int):
    """
    Full vanilla baseline training run for one seed.
    Returns dict of test metrics.
    """
    print(f"\n{'='*60}")
    print(f"VANILLA BASELINE — SEED {seed}")
    print(f"{'='*60}")

    set_seed(seed)

    # ── DataLoaders ───────────────────────────────────────────
    train_dataset = SingleTaskDataset(
        train_df["text"].values,
        train_df["sentiment_id"].values,
        tokenizer
    )
    val_dataset = SingleTaskDataset(
        val_df["text"].values,
        val_df["sentiment_id"].values,
        tokenizer
    )
    test_dataset = SingleTaskDataset(
        test_df["text"].values,
        test_df["sentiment_id"].values,
        tokenizer
    )
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

    # ── Model ─────────────────────────────────────────────────
    model = VanillaDistilBERT().to(device)

    # ── Loss ──────────────────────────────────────────────────
    s_weights = torch.tensor(sentiment_weights_vals, dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=s_weights)

    # ── Optimiser + Scheduler ─────────────────────────────────
    optimizer  = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler  = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = int(0.1 * total_steps),
        num_training_steps = total_steps
    )

    # ── Training loop ─────────────────────────────────────────
    best_val_f1      = 0.0
    best_model_state = None

    print(f"{'Epoch':<8}{'Train Loss':<14}{'Val F1':<12}{'Time'}")
    print("-" * 45)

    for epoch in range(EPOCHS):
        t0 = time.time()
        model.train()
        total_loss = 0

        for batch in train_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        # Validation
        model.eval()
        all_preds, all_labels_v = [], []
        with torch.no_grad():
            for batch in val_loader:
                logits = model(
                    batch['input_ids'].to(device),
                    batch['attention_mask'].to(device)
                )
                all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
                all_labels_v.extend(batch['label'].numpy())

        val_f1 = f1_score(all_labels_v, all_preds, average='macro')
        print(f"Epoch {epoch+1:<4} {total_loss/len(train_loader):<14.4f}{val_f1:<12.4f}{time.time()-t0:.1f}s")

        if val_f1 > best_val_f1:
            best_val_f1      = val_f1
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            print(f"         -> Best saved (Val F1: {val_f1:.4f})")

    # ── Test evaluation ───────────────────────────────────────
    model.load_state_dict(best_model_state)
    model.eval()
    all_preds, all_labels_t = [], []

    with torch.no_grad():
        for batch in test_loader:
            logits = model(
                batch['input_ids'].to(device),
                batch['attention_mask'].to(device)
            )
            all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
            all_labels_t.extend(batch['label'].numpy())

    macro_f1     = f1_score(all_labels_t, all_preds, average='macro')
    per_class_f1 = f1_score(all_labels_t, all_preds, average=None)
    accuracy     = (np.array(all_preds) == np.array(all_labels_t)).mean()

    print(f"\nTest Macro F1 : {macro_f1:.4f}")
    print(f"Test Accuracy : {accuracy:.4f}")
    print(classification_report(
        all_labels_t, all_preds,
        target_names=["negative", "neutral", "positive"]
    ))

    return {
        "seed"        : seed,
        "model"       : "Vanilla Baseline",
        "macro_f1"    : round(macro_f1, 4),
        "accuracy"    : round(accuracy, 4),
        "negative_f1" : round(per_class_f1[0], 4),
        "neutral_f1"  : round(per_class_f1[1], 4),
        "positive_f1" : round(per_class_f1[2], 4),
        "best_val_f1" : round(best_val_f1, 4)
    }

print("Vanilla training function defined.")

Vanilla training function defined.


### Cell 8 — Run All 6 Training Runs

Runs both models across all 3 seeds sequentially.  
Results are saved to Drive after **every single run** — if Colab disconnects  
mid-way, resume from Cell 9 which loads whatever is already saved.

Run order:
1. KD Pipeline — Seed 42
2. KD Pipeline — Seed 7
3. KD Pipeline — Seed 123
4. Vanilla Baseline — Seed 42
5. Vanilla Baseline — Seed 7
6. Vanilla Baseline — Seed 123

**Total expected runtime: ~55–65 minutes on T4 GPU.**

In [12]:
SEEDS        = [42, 7, 123]
RESULTS_PATH = f'{BASE}/statistical_significance.csv'
all_results  = []

# ── KD Pipeline runs ──────────────────────────────────────────
for seed in SEEDS:
    result = train_kd_pipeline(seed)
    all_results.append(result)
    # Save after every run
    pd.DataFrame(all_results).to_csv(RESULTS_PATH, index=False)
    print(f"\nCheckpoint saved after KD seed {seed}.")

# ── Vanilla Baseline runs ─────────────────────────────────────
for seed in SEEDS:
    result = train_vanilla_baseline(seed)
    all_results.append(result)
    # Save after every run
    pd.DataFrame(all_results).to_csv(RESULTS_PATH, index=False)
    print(f"\nCheckpoint saved after Vanilla seed {seed}.")

print("\nAll 6 runs complete.")


KD PIPELINE — SEED 42
Seed set to 42.


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch   Train Loss    Val F1      Time
---------------------------------------------
Epoch 1    0.7432        0.7788      47.5s
         -> Best saved (Val F1: 0.7788)
Epoch 2    0.3346        0.7977      47.3s
         -> Best saved (Val F1: 0.7977)
Epoch 3    0.1803        0.8472      47.2s
         -> Best saved (Val F1: 0.8472)
Epoch 4    0.0981        0.8578      47.5s
         -> Best saved (Val F1: 0.8578)
Epoch 5    0.0600        0.8484      47.3s

Test Macro F1 : 0.8056
Test Accuracy : 0.8206
              precision    recall  f1-score   support

    negative       0.74      0.90      0.81        61
     neutral       0.86      0.85      0.86       288
    positive       0.77      0.72      0.75       136

    accuracy                           0.82       485
   macro avg       0.79      0.82      0.81       485
weighted avg       0.82      0.82      0.82       485


Checkpoint saved after Vanilla seed 7.

VANILLA BASELINE — SEED 123
Seed set to 123.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch   Train Loss    Val F1      Time
---------------------------------------------
Epoch 1    0.7260        0.7705      47.3s
         -> Best saved (Val F1: 0.7705)
Epoch 2    0.3357        0.8046      47.4s
         -> Best saved (Val F1: 0.8046)
Epoch 3    0.1938        0.8381      47.2s
         -> Best saved (Val F1: 0.8381)
Epoch 4    0.0980        0.8312      47.2s
Epoch 5    0.0566        0.8285      47.3s

Test Macro F1 : 0.8021
Test Accuracy : 0.8186
              precision    recall  f1-score   support

    negative       0.73      0.85      0.79        61
     neutral       0.88      0.82      0.85       288
    positive       0.74      0.79      0.77       136

    accuracy                           0.82       485
   macro avg       0.79      0.82      0.80       485
weighted avg       0.82      0.82      0.82       485


Checkpoint saved after Vanilla seed 123.

All 6 runs complete.


### Cell 9 — Statistical Summary Table

Loads all results from Drive and computes `mean ± std` for every metric  
across the 3 seeds for each model.

This is the **final reportable result** — the format used in all ML papers.

If Cell 8 was interrupted, this cell loads whatever runs completed so far  
and computes statistics on the available runs.

In [13]:
RESULTS_PATH = f'{BASE}/statistical_significance.csv'

# Load all results
results_df = pd.read_csv(RESULTS_PATH)

print("RAW RESULTS — ALL RUNS")
print("=" * 75)
print(results_df.to_string(index=False))

# Compute mean and std per model
metrics = ["macro_f1", "accuracy", "negative_f1", "neutral_f1", "positive_f1"]

print("\n\nSTATISTICAL SUMMARY (mean +/- std across 3 seeds)")
print("=" * 65)
print(f"{'Metric':<20} {'KD Pipeline':<28} {'Vanilla Baseline':<28}")
print("-" * 65)

for metric in metrics:
    kd_vals  = results_df[results_df["model"] == "KD Pipeline"][metric]
    van_vals = results_df[results_df["model"] == "Vanilla Baseline"][metric]

    kd_str  = f"{kd_vals.mean():.4f} +/- {kd_vals.std():.4f}"
    van_str = f"{van_vals.mean():.4f} +/- {van_vals.std():.4f}"

    print(f"{metric:<20} {kd_str:<28} {van_str:<28}")

print("=" * 65)

# Gap with uncertainty
kd_f1  = results_df[results_df["model"] == "KD Pipeline"]["macro_f1"]
van_f1 = results_df[results_df["model"] == "Vanilla Baseline"]["macro_f1"]

gap_mean = van_f1.mean() - kd_f1.mean()
gap_std  = np.sqrt(van_f1.std()**2 + kd_f1.std()**2)

print(f"\nMacro F1 gap (vanilla - KD) : {gap_mean:+.4f} +/- {gap_std:.4f}")
print(f"Seeds used                  : {results_df['seed'].unique().tolist()}")
print(f"Runs per model              : {len(kd_f1)}")

RAW RESULTS — ALL RUNS
 seed            model  macro_f1  accuracy  negative_f1  neutral_f1  positive_f1  best_val_f1
   42      KD Pipeline    0.7582    0.7918       0.7704      0.8491       0.6552       0.7987
    7      KD Pipeline    0.7559    0.7918       0.7500      0.8482       0.6695       0.8060
  123      KD Pipeline    0.7477    0.7814       0.7463      0.8358       0.6609       0.7965
   42 Vanilla Baseline    0.8202    0.8330       0.8358      0.8682       0.7566       0.8420
    7 Vanilla Baseline    0.8056    0.8206       0.8148      0.8566       0.7452       0.8578
  123 Vanilla Baseline    0.8021    0.8186       0.7879      0.8525       0.7660       0.8381


STATISTICAL SUMMARY (mean +/- std across 3 seeds)
Metric               KD Pipeline                  Vanilla Baseline            
-----------------------------------------------------------------
macro_f1             0.7539 +/- 0.0055            0.8093 +/- 0.0096           
accuracy             0.7883 +/- 0.0060     

### Cell 10 — Paper-Ready Results Table

Formats the statistical summary into a clean table  
ready to paste directly into the research paper or report.

Also saves the final summary CSV to Drive for GitHub commit.

In [14]:
RESULTS_PATH = f'{BASE}/statistical_significance.csv'
results_df   = pd.read_csv(RESULTS_PATH)
metrics      = ["macro_f1", "accuracy", "negative_f1", "neutral_f1", "positive_f1"]

# Build summary rows
summary_rows = []
for metric in metrics:
    kd_vals  = results_df[results_df["model"] == "KD Pipeline"][metric]
    van_vals = results_df[results_df["model"] == "Vanilla Baseline"][metric]
    summary_rows.append({
        "metric"           : metric,
        "kd_mean"          : round(kd_vals.mean(),  4),
        "kd_std"           : round(kd_vals.std(),   4),
        "vanilla_mean"     : round(van_vals.mean(), 4),
        "vanilla_std"      : round(van_vals.std(),  4),
        "kd_str"           : f"{kd_vals.mean():.4f} +/- {kd_vals.std():.4f}",
        "vanilla_str"      : f"{van_vals.mean():.4f} +/- {van_vals.std():.4f}",
    })

summary_df = pd.DataFrame(summary_rows)

# Save to Drive
SUMMARY_PATH = f'{BASE}/statistical_summary.csv'
summary_df.to_csv(SUMMARY_PATH, index=False)

# Print paper-ready format
print("PAPER-READY RESULTS TABLE")
print("(mean +/- std over 3 seeds: 42, 7, 123)")
print("=" * 65)
print(f"{'Metric':<15} {'KD Pipeline':<28} {'Vanilla Baseline':<28}")
print("-" * 65)

label_map = {
    "macro_f1"    : "Macro F1",
    "accuracy"    : "Accuracy",
    "negative_f1" : "Negative F1",
    "neutral_f1"  : "Neutral F1",
    "positive_f1" : "Positive F1"
}

for row in summary_rows:
    print(f"{label_map[row['metric']]:<15} {row['kd_str']:<28} {row['vanilla_str']:<28}")

print("=" * 65)
print(f"\nNote: Seeds = [42, 7, 123] | Epochs = 5 | lr = 2e-5 | batch = 16")
print(f"\nFiles saved to Drive:")
print(f"  {RESULTS_PATH}")
print(f"  {SUMMARY_PATH}")

PAPER-READY RESULTS TABLE
(mean +/- std over 3 seeds: 42, 7, 123)
Metric          KD Pipeline                  Vanilla Baseline            
-----------------------------------------------------------------
Macro F1        0.7539 +/- 0.0055            0.8093 +/- 0.0096           
Accuracy        0.7883 +/- 0.0060            0.8241 +/- 0.0078           
Negative F1     0.7556 +/- 0.0130            0.8128 +/- 0.0240           
Neutral F1      0.8444 +/- 0.0074            0.8591 +/- 0.0081           
Positive F1     0.6619 +/- 0.0072            0.7559 +/- 0.0104           

Note: Seeds = [42, 7, 123] | Epochs = 5 | lr = 2e-5 | batch = 16

Files saved to Drive:
  /content/drive/MyDrive/KD_Project/statistical_significance.csv
  /content/drive/MyDrive/KD_Project/statistical_summary.csv
